# MIT 805 Part 2 — Kudzai's threads

**Central question (shared):** Where, when and for whom does New York's ridehail
market fail to match supply to demand — and how did congestion pricing change that?

**This notebook covers:**

| § | Content | Rubric criterion |
|---|---|---|
| 1–3 | Setup, load, column pruning | — |
| 4 | Shared zone×hour×month aggregate (Savannah reads this too) | 2, 3, 4 |
| 5 | Dual implementation: RDD `reduceByKey` vs DataFrame `groupBy` | 2, 3, 4, 5 |
| 6 | Distributed execution evidence: plans, DAG, shuffle bytes | 5 |
| 7 | Thread A — congestion pricing, 2024 vs 2025 | 6, 8 |
| 8 | Thread B — supply-side service failure | 6, 8 |
| 9 | Figures | 7 |
| 10 | Export numbers for the report | — |

Outputs land in `output/p2/` and `figures/p2/`.

**Run order matters.** §4 is the expensive job (20–40 min).

## 1. Environment

In [1]:
import sys, os
os.environ["SPARK_AUTH_SOCKET_TIMEOUT"] = "300"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
print("worker timeout raised; python:", sys.executable)





import os, json, time, math
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("cwd:", os.getcwd())

for d in ["output/p2", "figures/p2", "spark-events", "tmp"]:
    os.makedirs(d, exist_ok=True)

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 130, "savefig.bbox": "tight",
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 9,
                     "figure.facecolor": "white", "savefig.facecolor": "white"})
GB = 1024 ** 3
print("ready")

worker timeout raised; python: D:\805\.venv311\Scripts\python.exe
cwd: D:\805
ready


In [2]:
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("MIT805-Part2-HVFHV")
    .master("local[8]")
    .config("spark.driver.memory", "32g")
    .config("spark.sql.shuffle.partitions", "96")
    .config("spark.sql.parquet.mergeSchema", "true")
    .config("spark.sql.session.timeZone", "America/New_York")
    .config("spark.local.dir", "D:/805/tmp")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///D:/805/spark-events")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| parallelism:", spark.sparkContext.defaultParallelism)
print("Spark UI:", spark.sparkContext.uiWebUrl)
print("\n>>> Open the UI in a browser NOW and keep the tab open — it dies with the session.")

Spark 3.5.1 | parallelism: 8
Spark UI: http://MAMPC020:4040

>>> Open the UI in a browser NOW and keep the tab open — it dies with the session.


## 2. Data

We compare calendar 2024 (baseline) against calendar 2025 (congestion pricing in
force from 5 January 2025). Twenty-four monthly files, roughly 480 million records.

In [3]:
import requests

RAW = r"D:\805\data\raw"
os.makedirs(RAW, exist_ok=True)
BASE = "https://d37ci6vzurychx.cloudfront.net/trip-data"

def fname(y, m): return f"fhvhv_tripdata_{y:04d}-{m:02d}.parquet"

def fetch(y, m):
    dest = os.path.join(RAW, fname(y, m))
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        return
    url = f"{BASE}/{fname(y, m)}"
    print(f"  downloading {fname(y, m)} ...", end="", flush=True)
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(dest + ".part", "wb") as f:
            for chunk in r.iter_content(1024 * 1024):
                f.write(chunk)
    os.replace(dest + ".part", dest)
    print(f" {os.path.getsize(dest)/1024/1024:.0f} MB")

for m in range(1, 13):
    fetch(2024, m)
print("2024 complete")

on_disk = sum(os.path.getsize(os.path.join(RAW, f)) for f in os.listdir(RAW))
print(f"data/raw total: {on_disk/GB:.2f} GB")

2024 complete
data/raw total: 11.43 GB


In [4]:
# Column pruning: we read 14 of 25 columns. This is a Parquet/columnar win worth
# reporting - the scan never touches the other 11, which is visible in the
# physical plan as ReadSchema.
COLS_2025 = ["hvfhs_license_num", "request_datetime", "pickup_datetime",
             "dropoff_datetime", "PULocationID", "DOLocationID", "trip_miles",
             "trip_time", "base_passenger_fare", "driver_pay", "tips",
             "shared_request_flag", "shared_match_flag", "cbd_congestion_fee"]
COLS_2024 = [c for c in COLS_2025 if c != "cbd_congestion_fee"]

p24 = [f"{RAW}/{fname(2024, m)}".replace("\\", "/") for m in range(1, 13)]
p25 = [f"{RAW}/{fname(2025, m)}".replace("\\", "/") for m in range(1, 13)]

df24 = (spark.read.parquet(*p24).select(*COLS_2024)
        .withColumn("cbd_congestion_fee", F.lit(0.0)))
df25 = spark.read.parquet(*p25).select(*COLS_2025)

df = df24.unionByName(df25).withColumn("year", F.year("pickup_datetime"))
print("columns:", len(df.columns))
df.printSchema()

columns: 15
root
 |-- hvfhs_license_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- year: integer (nullable = true)



In [5]:
# Cleaning rules carried over from Part 1, applied identically to both years so
# the before/after comparison is like-for-like.
clean = df.filter(
    (F.col("trip_miles") > 0) &
    (F.col("trip_time") > 0) & (F.col("trip_time") <= 6 * 3600) &
    (F.col("base_passenger_fare") >= 0) &
    (F.col("dropoff_datetime") > F.col("pickup_datetime")) &
    (~F.col("PULocationID").isin(264, 265)) &
    (~F.col("DOLocationID").isin(264, 265)) &
    (F.col("year").isin(2024, 2025))
)

t0 = time.time()
by_year = clean.groupBy("year").count().orderBy("year").toPandas()
print(f"counted in {time.time()-t0:.0f}s")
print(by_year)
n_total = int(by_year["count"].sum())
print(f"total retained: {n_total:,}")

counted in 125s
   year      count
0  2024  229391984
1  2025  232463392
total retained: 461,855,376


## 3. Deriving the congestion-pricing zone set

The fee applies to trips touching the Manhattan central business district, but we
do not hardcode the boundary. Instead we derive it from the data: in 2025, group
by zone and compute the share of trips carrying a non-zero fee. Zones where that
share is high are CBD zones. This is reproducible, auditable, and avoids baking a
policy assumption into the analysis.

In [6]:
zones = (spark.read.option("header", True)
         .csv(f"{RAW}/taxi_zone_lookup.csv".replace("\\", "/"))
         .withColumn("LocationID", F.col("LocationID").cast("int")))

charged = (clean.filter(F.col("year") == 2025)
           .groupBy("DOLocationID")
           .agg(F.count("*").alias("trips"),
                F.avg((F.col("cbd_congestion_fee") > 0).cast("double")).alias("charge_rate"))
           .filter(F.col("trips") > 50000)
           .join(zones, F.col("DOLocationID") == F.col("LocationID"))
           .select("DOLocationID", "Zone", "Borough", "trips", "charge_rate")
           .orderBy(F.desc("charge_rate")))

cp = charged.toPandas()
cp.to_csv("output/p2/zone_charge_rate.csv", index=False)
print(cp.head(25).to_string(index=False))
print("\n--- distribution of charge_rate ---")
print(cp["charge_rate"].describe())

 DOLocationID                          Zone   Borough   trips  charge_rate
          161                Midtown Center Manhattan 2482591     0.987747
          162                  Midtown East Manhattan 1925807     0.987004
          234                      Union Sq Manhattan 2195676     0.985907
          163                 Midtown North Manhattan 1648058     0.985801
          125                     Hudson Sq Manhattan  737239     0.985382
          100              Garment District Manhattan 1054227     0.985288
          230     Times Sq/Theatre District Manhattan 2423941     0.985281
          211                          SoHo Manhattan 1194567     0.985162
          113       Greenwich Village North Manhattan 1150584     0.984966
           68                  East Chelsea Manhattan 2483451     0.984923
          186  Penn Station/Madison Sq West Manhattan 1795272     0.984089
          164                 Midtown South Manhattan 2005121     0.984042
          144           L

In [7]:
# Threshold: zones where most arriving trips are charged are inside the CBD.
# Inspect the output above before fixing this - the distribution should be
# strongly bimodal, which itself is evidence the boundary is real.
CBD_THRESHOLD = 0.80
cbd_ids = sorted(cp.loc[cp.charge_rate >= CBD_THRESHOLD, "DOLocationID"].astype(int).tolist())
print(f"{len(cbd_ids)} CBD zones at threshold {CBD_THRESHOLD}:")
print(cp.loc[cp.DOLocationID.isin(cbd_ids), ["Zone", "Borough", "charge_rate"]]
        .to_string(index=False))

# Boundary proxy: Manhattan zones outside the CBD. Without the taxi-zone
# shapefile we cannot compute true adjacency, so we treat non-CBD Manhattan as
# the displacement candidate and say so in the report.
manh_ids = [int(r.LocationID) for r in
            zones.filter(F.col("Borough") == "Manhattan").select("LocationID").collect()]
boundary_ids = sorted(set(manh_ids) - set(cbd_ids))
print(f"\n{len(boundary_ids)} non-CBD Manhattan (boundary proxy) zones")

json.dump({"cbd_threshold": CBD_THRESHOLD, "cbd_zone_ids": cbd_ids,
           "boundary_zone_ids": boundary_ids},
          open("output/p2/cbd_zones.json", "w"), indent=2)

38 CBD zones at threshold 0.8:
                          Zone   Borough  charge_rate
                Midtown Center Manhattan     0.987747
                  Midtown East Manhattan     0.987004
                      Union Sq Manhattan     0.985907
                 Midtown North Manhattan     0.985801
                     Hudson Sq Manhattan     0.985382
              Garment District Manhattan     0.985288
     Times Sq/Theatre District Manhattan     0.985281
                          SoHo Manhattan     0.985162
       Greenwich Village North Manhattan     0.984966
                  East Chelsea Manhattan     0.984923
  Penn Station/Madison Sq West Manhattan     0.984089
                 Midtown South Manhattan     0.984042
           Little Italy/NoLiTa Manhattan     0.983632
                   Murray Hill Manhattan     0.983491
       Greenwich Village South Manhattan     0.983481
                  West Village Manhattan     0.982375
                      Gramercy Manhattan     0.9818

In [8]:
cbd_b = F.broadcast(spark.createDataFrame([(i,) for i in cbd_ids], ["cbd_id"]))

tagged = (clean
    .withColumn("pu_cbd", F.col("PULocationID").isin(cbd_ids))
    .withColumn("do_cbd", F.col("DOLocationID").isin(cbd_ids))
    .withColumn("cbd_trip", F.col("pu_cbd") | F.col("do_cbd"))
    .withColumn("wait_s", F.unix_timestamp("pickup_datetime")
                        - F.unix_timestamp("request_datetime"))
    .withColumn("wait_valid", (F.col("wait_s") >= 0) & (F.col("wait_s") <= 3600))
).cache()

print(tagged.groupBy("year", "cbd_trip").count().orderBy("year", "cbd_trip").toPandas())

   year  cbd_trip      count
0  2024     False  149494083
1  2024      True   79897901
2  2025     False  156073390
3  2025      True   76390002


## 4. Shared aggregate — the MapReduce core

One job, written once, consumed by both group members. Keys are
zone × zone × year × month × day-of-week × hour; measures are **sums**, never
averages, so any downstream regrouping re-aggregates correctly.

This is the map → shuffle → reduce backbone of the analysis:
`withColumn` projections are the map side, `groupBy` forces the shuffle on the
composite key, and the `agg` expressions are the reduction.

In [9]:
t0 = time.time()

agg = (tagged
    .withColumn("month", F.month("pickup_datetime"))
    .withColumn("dow", F.dayofweek("pickup_datetime"))
    .withColumn("hour", F.hour("pickup_datetime"))
    .groupBy("PULocationID", "DOLocationID", "year", "month", "dow", "hour")
    .agg(
        F.count("*").alias("trips"),
        F.sum("base_passenger_fare").alias("sum_fare"),
        F.sum("driver_pay").alias("sum_driver_pay"),
        F.sum("tips").alias("sum_tips"),
        F.sum("cbd_congestion_fee").alias("sum_cbd_fee"),
        F.sum("trip_miles").alias("sum_miles"),
        F.sum("trip_time").alias("sum_time"),
        F.sum(F.when(F.col("wait_valid"), F.col("wait_s"))).alias("sum_wait"),
        F.count(F.when(F.col("wait_valid"), 1)).alias("wait_n"),
        F.count(F.when(F.col("shared_request_flag") == "Y", 1)).alias("shared_req"),
        F.count(F.when((F.col("shared_request_flag") == "Y") &
                       (F.col("shared_match_flag") == "Y"), 1)).alias("shared_match"),
    ))

agg.write.mode("overwrite").parquet("output/p2/agg_zone_hour_month.parquet")
print(f"shared aggregate written in {time.time()-t0:.0f}s")

A = spark.read.parquet("output/p2/agg_zone_hour_month.parquet").cache()
print(f"{A.count():,} aggregate rows")
A.printSchema()

shared aggregate written in 555s
73,973,527 aggregate rows
root
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- dow: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- trips: long (nullable = true)
 |-- sum_fare: double (nullable = true)
 |-- sum_driver_pay: double (nullable = true)
 |-- sum_tips: double (nullable = true)
 |-- sum_cbd_fee: double (nullable = true)
 |-- sum_miles: double (nullable = true)
 |-- sum_time: long (nullable = true)
 |-- sum_wait: long (nullable = true)
 |-- wait_n: long (nullable = true)
 |-- shared_req: long (nullable = true)
 |-- shared_match: long (nullable = true)



## 5. Dual implementation — RDD vs DataFrame

The same computation (trips and mean wait by pickup zone and hour) written twice:
once as an explicit low-level MapReduce job, once with the DataFrame API. This is
what lets us compare Spark's DAG execution against the classical
Map → Shuffle → Reduce model with our own evidence rather than assertion.

In [12]:
# --- (a) RDD: explicit Map -> Shuffle -> Reduce ------------------------------
t0 = time.time()
rdd_out = (src.rdd
    # MAP: one record -> one (key, value) pair
    .map(lambda r: ((r.PULocationID, r.pickup_datetime.hour),
                    (1, (r.pickup_datetime - r.request_datetime).total_seconds())))
    # SHUFFLE + REDUCE: reduceByKey combines locally on each partition before
    # shuffling, which is the Spark analogue of a Hadoop Combiner.
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    # MAP: finalise partial sums into a mean
    .map(lambda kv: (kv[0][0], kv[0][1], kv[1][0], kv[1][1] / kv[1][0]))
    .collect())
rdd_secs = time.time() - t0
print(f"RDD  reduceByKey : {rdd_secs:6.1f}s  -> {len(rdd_out):,} keys")
print("partitions shuffled:", src.rdd.getNumPartitions())

RDD  reduceByKey :   12.7s  -> 6,143 keys
partitions shuffled: 8


In [13]:
# --- (b) DataFrame: same result, Catalyst-optimised ---------------------------
t0 = time.time()
df_out = (src
    .withColumn("wait_s", F.unix_timestamp("pickup_datetime")
                          - F.unix_timestamp("request_datetime"))
    .groupBy("PULocationID", F.hour("pickup_datetime").alias("hour"))
    .agg(F.count("*").alias("trips"), F.avg("wait_s").alias("mean_wait_s"))
    .collect())
df_secs = time.time() - t0
print(f"DataFrame groupBy: {df_secs:6.1f}s  -> {len(df_out):,} keys")

speedup = rdd_secs / df_secs if df_secs else float("nan")
print(f"\nDataFrame is {speedup:.2f}x faster on identical input ({n_src:,} rows)")

# verify both paths agree
r = sorted(rdd_out)[:3]
d = sorted([(x[0], x[1], x[2], x[3]) for x in df_out])[:3]
print("\nRDD  sample:", [(a, b, c, round(e, 2)) for a, b, c, e in r])
print("DF   sample:", [(a, b, c, round(e, 2)) for a, b, c, e in d])

rdd_map = {(a, b): (c, e) for a, b, c, e in rdd_out}
df_map  = {(x[0], x[1]): (x[2], x[3]) for x in df_out}
common  = set(rdd_map) & set(df_map)
max_trip_diff = max(abs(rdd_map[k][0] - df_map[k][0]) for k in common)
max_wait_diff = max(abs(rdd_map[k][1] - df_map[k][1]) for k in common)
print(f"\nkeys compared : {len(common):,}")
print(f"max trip diff : {max_trip_diff}")
print(f"max wait diff : {max_wait_diff:.9f} s  (floating point only)")

os.makedirs("output/p2", exist_ok=True)
json.dump({"rows_benchmarked": int(n_src),
           "partitions": int(src.rdd.getNumPartitions()),
           "rdd_seconds": round(rdd_secs, 1),
           "df_seconds": round(df_secs, 1),
           "speedup": round(speedup, 2),
           "keys": len(rdd_out)},
          open("output/p2/dual_implementation_timing.json", "w"), indent=2)
print("\nwritten: output/p2/dual_implementation_timing.json")

DataFrame groupBy:    1.9s  -> 6,143 keys

DataFrame is 6.56x faster on identical input (1,964,006 rows)

RDD  sample: [(2, 11, 1, 216.0), (2, 12, 2, 193.0), (2, 16, 1, 1023.0)]
DF   sample: [(2, 11, 1, 216.0), (2, 12, 2, 193.0), (2, 16, 1, 1023.0)]

keys compared : 6,143
max trip diff : 0
max wait diff : 0.000000000 s  (floating point only)

written: output/p2/dual_implementation_timing.json


## 6. Distributed execution evidence

Three artefacts for the report: the physical plan, the join-strategy comparison,
and the Spark UI DAG. Capture screenshots of the UI **while this session is
alive** — stages, the DAG visualisation, and the shuffle read/write columns.

In [14]:
plan_df = (tagged.filter(F.col("wait_valid") & (F.col("year") == 2025))
    .groupBy("PULocationID", F.hour("pickup_datetime").alias("hour"))
    .agg(F.count("*").alias("trips"), F.avg("wait_s").alias("mean_wait_s")))

print("=" * 78)
print("PHYSICAL PLAN - DataFrame aggregation")
print("=" * 78)
plan_df.explain("formatted")

PHYSICAL PLAN - DataFrame aggregation
== Physical Plan ==
AdaptiveSparkPlan (21)
+- HashAggregate (20)
   +- Exchange (19)
      +- HashAggregate (18)
         +- Project (17)
            +- Filter (16)
               +- InMemoryTableScan (1)
                     +- InMemoryRelation (2)
                           +- Union (15)
                              :- * Project (8)
                              :  +- * Project (7)
                              :     +- * Project (6)
                              :        +- * Filter (5)
                              :           +- * ColumnarToRow (4)
                              :              +- Scan parquet  (3)
                              +- * Project (14)
                                 +- * Project (13)
                                    +- * Project (12)
                                       +- * Filter (11)
                                          +- * ColumnarToRow (10)
                                             +- Scan parquet

In [15]:
# Capture the plan to a text file for the appendix.
plan_text = plan_df._jdf.queryExecution().toString()
with open("output/p2/physical_plan_groupby.txt", "w", encoding="utf-8") as f:
    f.write(plan_text)
print(plan_text[:2500])

== Parsed Logical Plan ==
'Aggregate ['PULocationID, hour('pickup_datetime, None) AS hour#2916], ['PULocationID, hour('pickup_datetime, None) AS hour#2916, count(1) AS trips#2938L, avg('wait_s) AS mean_wait_s#2940]
+- Filter (wait_valid#363 AND (year#156 = 2025))
   +- Project [hvfhs_license_num#0, request_datetime#3, pickup_datetime#5, dropoff_datetime#6, PULocationID#7, DOLocationID#8, trip_miles#9, trip_time#10L, base_passenger_fare#11, driver_pay#18, tips#17, shared_request_flag#19, shared_match_flag#20, cbd_congestion_fee#62, year#156, pu_cbd#289, do_cbd#306, cbd_trip#324, wait_s#343L, ((wait_s#343L >= cast(0 as bigint)) AND (wait_s#343L <= cast(3600 as bigint))) AS wait_valid#363]
      +- Project [hvfhs_license_num#0, request_datetime#3, pickup_datetime#5, dropoff_datetime#6, PULocationID#7, DOLocationID#8, trip_miles#9, trip_time#10L, base_passenger_fare#11, driver_pay#18, tips#17, shared_request_flag#19, shared_match_flag#20, cbd_congestion_fee#62, year#156, pu_cbd#289, do_cbd

In [16]:
# Join strategy: broadcast vs sort-merge. The zone lookup is 265 rows, so Spark
# broadcasts it and avoids shuffling the 480M-row side entirely.
small = zones.select(F.col("LocationID").alias("zid"), "Zone", "Borough")

print("=" * 78); print("BROADCAST JOIN"); print("=" * 78)
(tagged.join(F.broadcast(small), tagged.PULocationID == small.zid)
       .groupBy("Borough").count()).explain("formatted")

print("\n" + "=" * 78); print("SORT-MERGE JOIN (broadcast disabled)"); print("=" * 78)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
(tagged.join(small, tagged.PULocationID == small.zid)
       .groupBy("Borough").count()).explain("formatted")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
print("\n(threshold restored)")

BROADCAST JOIN
== Physical Plan ==
AdaptiveSparkPlan (26)
+- HashAggregate (25)
   +- Exchange (24)
      +- HashAggregate (23)
         +- Project (22)
            +- BroadcastHashJoin Inner BuildRight (21)
               :- Filter (16)
               :  +- InMemoryTableScan (1)
               :        +- InMemoryRelation (2)
               :              +- Union (15)
               :                 :- * Project (8)
               :                 :  +- * Project (7)
               :                 :     +- * Project (6)
               :                 :        +- * Filter (5)
               :                 :           +- * ColumnarToRow (4)
               :                 :              +- Scan parquet  (3)
               :                 +- * Project (14)
               :                    +- * Project (13)
               :                       +- * Project (12)
               :                          +- * Filter (11)
               :                             +- * Co

In [17]:
# Stage / task metrics for the report. Also note the Spark UI URL again.
sc = spark.sparkContext
print("Spark UI           :", sc.uiWebUrl)
print("application id     :", sc.applicationId)
print("default parallelism:", sc.defaultParallelism)
print("shuffle partitions :", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE enabled        :", spark.conf.get("spark.sql.adaptive.enabled"))
print("\nIn the UI capture: (1) the DAG visualisation for the groupBy job,")
print("(2) the stage list showing shuffle read/write bytes, (3) the task timeline.")

Spark UI           : http://MAMPC020:4040
application id     : local-1791374291981
default parallelism: 8
shuffle partitions : 96
AQE enabled        : true

In the UI capture: (1) the DAG visualisation for the groupBy job,
(2) the stage list showing shuffle read/write bytes, (3) the task timeline.


## 7. Thread A — Congestion pricing, 2024 vs 2025

The fee took effect 5 January 2025. We compare CBD-touching trips against
non-CBD trips across the two years. This is a difference-in-differences *shape*,
not a causal identification: 2024 and 2025 differ in more than the policy, and
the report must say so.

In [18]:
zb = zones.select(F.col("LocationID").alias("zid"), F.col("Borough").alias("pu_boro"))

monthly = (A
    .withColumn("cbd_trip", F.col("PULocationID").isin(cbd_ids) |
                            F.col("DOLocationID").isin(cbd_ids))
    .groupBy("year", "month", "cbd_trip")
    .agg(F.sum("trips").alias("trips"),
         F.sum("sum_fare").alias("fare"),
         F.sum("sum_cbd_fee").alias("cbd_fee"),
         F.sum("sum_miles").alias("miles"),
         F.sum("sum_wait").alias("wait"), F.sum("wait_n").alias("wait_n"))
    .orderBy("year", "month", "cbd_trip").toPandas())

monthly["mean_fare"] = monthly["fare"] / monthly["trips"]
monthly["mean_miles"] = monthly["miles"] / monthly["trips"]
monthly["mean_wait_s"] = monthly["wait"] / monthly["wait_n"]
monthly.to_csv("output/p2/monthly_cbd_vs_noncbd.csv", index=False)
monthly

,year,month,cbd_trip,trips,fare,cbd_fee,miles,wait,wait_n,mean_fare,mean_miles,mean_wait_s
0,2024,1,False,12218405,2.368578e+08,0.0,4.986624e+07,3183698797,12107113,19.385330,4.081240,262.961021
1,2024,1,True,6700307,1.856808e+08,0.0,3.282298e+07,1599490414,6634932,27.712281,4.898728,241.071109
2,2024,2,False,11965642,2.340472e+08,0.0,4.972047e+07,3143645787,11842821,19.559940,4.155269,265.447378
3,2024,2,True,6627991,1.871554e+08,0.0,3.315011e+07,1654744888,6564267,28.237123,5.001532,252.083727
4,2024,3,False,13362511,2.741225e+08,0.0,5.515776e+07,3681868805,13262431,20.514299,4.127799,277.616434
5,2024,3,True,7080743,2.127276e+08,0.0,3.610237e+07,1889193758,7010754,30.043116,5.098670,269.470838
6,2024,4,False,12265682,2.513690e+08,0.0,5.137726e+07,3355969629,12167971,20.493686,4.188700,275.803553
7,2024,4,True,6658095,2.032803e+08,0.0,3.415981e+07,1784091611,6593415,30.531305,5.130568,270.586883
8,2024,5,False,12880483,2.744479e+08,0.0,5.472425e+07,3544331150,12776227,21.307271,4.248618,277.416106
9,2024,5,True,6947973,2.243541e+08,0.0,3.656418e+07,1849508951,6878083,32.290589,5.262568,268.898900


In [19]:
piv = monthly.pivot_table(index=["year", "month"], columns="cbd_trip", values="trips")
piv.columns = ["non_cbd", "cbd"]
piv["cbd_share_pct"] = piv["cbd"] / (piv["cbd"] + piv["non_cbd"]) * 100
piv["ratio"] = piv["cbd"] / piv["non_cbd"]

base24 = piv.loc[2024]
base25 = piv.loc[2025]
print("2024 mean CBD share: %.2f%%" % base24["cbd_share_pct"].mean())
print("2025 mean CBD share: %.2f%%" % base25["cbd_share_pct"].mean())
print("change            : %+.2f pp" % (base25["cbd_share_pct"].mean()
                                        - base24["cbd_share_pct"].mean()))
print("\nCBD:non-CBD ratio 2024 %.4f -> 2025 %.4f (%+.1f%%)" % (
    base24["ratio"].mean(), base25["ratio"].mean(),
    (base25["ratio"].mean() / base24["ratio"].mean() - 1) * 100))
piv.to_csv("output/p2/cbd_share_by_month.csv")
piv

2024 mean CBD share: 34.84%
2025 mean CBD share: 32.88%
change            : -1.96 pp

CBD:non-CBD ratio 2024 0.5348 -> 2025 0.4902 (-8.3%)


non_cbd        cbd  cbd_share_pct     ratio
year month                                                
2024 1      12218405.0  6700307.0      35.416296  0.548378
     2      11965642.0  6627991.0      35.646562  0.553919
     3      13362511.0  7080743.0      34.636086  0.529896
     4      12265682.0  6658095.0      35.183753  0.542823
     5      12880483.0  6947973.0      35.040414  0.539419
     6      12574041.0  6681413.0      34.698808  0.531366
     7      11979767.0  6391885.0      34.792108  0.533557
     8      12090297.0  6224499.0      33.986177  0.514834
     9      11875259.0  6483042.0      35.313954  0.545928
     10     12338952.0  6787203.0      35.486500  0.550063
     11     12497855.0  6625471.0      34.646018  0.530129
     12     13445189.0  6689279.0      33.223023  0.497522
2025 1      13037401.0  6547516.0      33.431421  0.502210
     2      12241945.0  6289614.0      33.940016  0.513776
     3      13155884.0  6470215.0      32.967402  0.491811
     4      12514358.0  6360350.0      33.697740  0.508244
     5      13521471.0  6592451.0      32.775562  0.487554
     6      12826226.0  6107856.0      32.258527  0.476201
     7      12679341.0  6042533.0      32.275257  0.476565
     8      12572801.0  5773853.0      31.470877  0.459234
     9      12127130.0  6354070.0      34.381263  0.523955
     10     13399189.0  6915214.0      34.040941  0.516092
     11     13338824.0  6497729.0      32.756341  0.487129
     12     14658820.0  6438601.0      30.518427  0.439231

In [20]:
# Boundary displacement: did non-CBD Manhattan gain as CBD lost?
seg = (A.withColumn("seg",
        F.when(F.col("PULocationID").isin(cbd_ids) |
               F.col("DOLocationID").isin(cbd_ids), "cbd")
         .when(F.col("PULocationID").isin(boundary_ids) |
               F.col("DOLocationID").isin(boundary_ids), "boundary")
         .otherwise("outer"))
    .groupBy("year", "month", "seg").agg(F.sum("trips").alias("trips"))
    .orderBy("year", "month", "seg").toPandas())

sp = seg.pivot_table(index=["year", "month"], columns="seg", values="trips")
yr = sp.groupby(level=0).sum()
chg = (yr.loc[2025] / yr.loc[2024] - 1) * 100
print("Year-on-year change in trips by segment (%):")
print(chg.round(2).to_string())
seg.to_csv("output/p2/segment_monthly.csv", index=False)

Year-on-year change in trips by segment (%):
seg
boundary    0.79
cbd        -4.39
outer       5.26


In [21]:
# Fee incidence by pickup borough - the equity finding from Part 1, now measured.
inc = (A.filter(F.col("year") == 2025)
    .join(F.broadcast(zb), A.PULocationID == zb.zid)
    .groupBy("pu_boro")
    .agg(F.sum("trips").alias("trips"),
         F.sum("sum_fare").alias("fare"),
         F.sum("sum_cbd_fee").alias("fee"))
    .toPandas())
inc["mean_fare"] = inc["fare"] / inc["trips"]
inc["fee_per_trip"] = inc["fee"] / inc["trips"]
inc["fee_pct_of_fare"] = inc["fee"] / inc["fare"] * 100
inc["pct_trips_charged"] = inc["fee"] / 1.50 / inc["trips"] * 100
inc = inc.sort_values("fee_pct_of_fare", ascending=False)
inc.to_csv("output/p2/fee_incidence_by_borough.csv", index=False)
inc[["pu_boro", "trips", "mean_fare", "fee_per_trip",
     "fee_pct_of_fare", "pct_trips_charged"]].round(2)

,pu_boro,trips,mean_fare,fee_per_trip,fee_pct_of_fare,pct_trips_charged
5,Manhattan,85115412,29.03,1.14,3.94,76.28
3,Queens,49489947,25.72,0.17,0.67,11.48
0,Brooklyn,64586172,21.84,0.15,0.67,9.74
4,EWR,8,40.50,0.19,0.46,12.50
2,Bronx,29547926,18.79,0.06,0.30,3.74
1,Staten Island,3723927,20.90,0.03,0.15,2.14


## 8. Thread B — Supply-side service failure

Part 1 showed wait peaks at 04:00 and troughs at 10:00 — the inverse of demand.
Here we localise that: which zones, at which hours, fail worst relative to what
the rest of the city achieves at the same hour.

In [22]:
zh = (A.filter(F.col("year") == 2025)
    .groupBy("PULocationID", "hour")
    .agg(F.sum("trips").alias("trips"),
         F.sum("sum_wait").alias("sum_wait"),
         F.sum("wait_n").alias("wait_n"),
         F.sum("shared_req").alias("shared_req"),
         F.sum("shared_match").alias("shared_match"))
    .filter(F.col("wait_n") > 1000)
    .withColumn("mean_wait_s", F.col("sum_wait") / F.col("wait_n")))

# Citywide mean wait per hour, then a mismatch index = zone wait / citywide wait.
# A window over hour computes the benchmark without a second pass over the data.
w = Window.partitionBy("hour")
zh = (zh.withColumn("city_wait_s",
                    F.sum("sum_wait").over(w) / F.sum("wait_n").over(w))
        .withColumn("mismatch_index", F.col("mean_wait_s") / F.col("city_wait_s")))

zh_p = (zh.join(F.broadcast(zones), zh.PULocationID == zones.LocationID)
          .select("Zone", "Borough", "hour", "trips", "mean_wait_s",
                  "city_wait_s", "mismatch_index", "shared_req", "shared_match")
          .toPandas())
zh_p.to_csv("output/p2/zone_hour_wait.csv", index=False)
print(f"{len(zh_p):,} zone-hour cells")
print(zh_p.nlargest(15, "mismatch_index")[
    ["Zone", "Borough", "hour", "trips", "mean_wait_s", "mismatch_index"]]
    .round(2).to_string(index=False))

5,898 zone-hour cells
                               Zone       Borough  hour  trips  mean_wait_s  mismatch_index
       Flushing Meadows-Corona Park        Queens    23  16290       752.98            2.35
Breezy Point/Fort Tilden/Riis Beach        Queens    20   1201       641.18            2.35
       Flushing Meadows-Corona Park        Queens    22  20474       721.85            2.30
       Flushing Meadows-Corona Park        Queens     0  10346       713.98            2.27
Breezy Point/Fort Tilden/Riis Beach        Queens    19   1609       605.32            2.20
                        City Island         Bronx    23   2789       671.47            2.10
Breezy Point/Fort Tilden/Riis Beach        Queens    17   1704       597.86            2.07
Breezy Point/Fort Tilden/Riis Beach        Queens    18   1811       584.93            2.05
             Charleston/Tottenville Staten Island     2   1997       622.82            2.02
                    Randalls Island     Manhattan    22   

In [23]:
# Persistent service deserts: zones in the worst decile of mismatch across many hours.
thr = zh_p["mismatch_index"].quantile(0.90)
bad = (zh_p[zh_p.mismatch_index >= thr]
       .groupby(["Zone", "Borough"])
       .agg(bad_hours=("hour", "count"),
            mean_mismatch=("mismatch_index", "mean"),
            trips=("trips", "sum")).reset_index()
       .sort_values(["bad_hours", "mean_mismatch"], ascending=False))
bad.to_csv("output/p2/service_deserts.csv", index=False)
print(f"worst-decile threshold: mismatch_index >= {thr:.2f}")
print(bad.head(15).round(2).to_string(index=False))

worst-decile threshold: mismatch_index >= 1.30
                             Zone       Borough  bad_hours  mean_mismatch  trips
                  Hammels/Arverne        Queens         24           1.61 370617
           Charleston/Tottenville Staten Island         24           1.61 104219
                    Rockaway Park        Queens         24           1.55 169166
                     Far Rockaway        Queens         24           1.54 439550
                      Great Kills Staten Island         22           1.51 151597
Eltingville/Annadale/Prince's Bay Staten Island         22           1.47 100112
                    Arden Heights Staten Island         20           1.50  65169
           New Dorp/Midland Beach Staten Island         20           1.48 125543
              Grymes Hill/Clifton Staten Island         20           1.44 164979
                       Douglaston        Queens         20           1.43 105621
                      City Island         Bronx         18    

In [24]:
# Does density help or hurt? Part 1 implied the relationship is negative.
sub = zh_p[zh_p.trips > 0].copy()
sub["log_trips"] = sub["trips"].apply(lambda x: math.log10(x))
corr = sub[["log_trips", "mean_wait_s"]].corr().iloc[0, 1]
print(f"corr(log10 trips, mean wait) = {corr:.3f}")

by_hour = (sub.groupby("hour")
             .apply(lambda g: g[["log_trips", "mean_wait_s"]].corr().iloc[0, 1])
             .rename("corr").reset_index())
by_hour.to_csv("output/p2/density_wait_corr_by_hour.csv", index=False)
print(by_hour.round(3).to_string(index=False))

corr(log10 trips, mean wait) = -0.472
 hour   corr
    0 -0.522
    1 -0.493
    2 -0.447
    3 -0.425
    4 -0.350
    5 -0.376
    6 -0.308
    7 -0.288
    8 -0.240
    9 -0.316
   10 -0.454
   11 -0.484
   12 -0.448
   13 -0.466
   14 -0.444
   15 -0.456
   16 -0.479
   17 -0.461
   18 -0.465
   19 -0.567
   20 -0.617
   21 -0.542
   22 -0.488
   23 -0.571


In [25]:
# Shared-ride matching failure by hour.
sm = (zh_p.groupby("hour")[["shared_req", "shared_match"]].sum().reset_index())
sm["match_rate_pct"] = sm["shared_match"] / sm["shared_req"] * 100
sm.to_csv("output/p2/shared_match_by_hour.csv", index=False)
print(sm.round(2).to_string(index=False))

 hour  shared_req  shared_match  match_rate_pct
    0      396458        263152           66.38
    1      273654        180812           66.07
    2      204404        129288           63.25
    3      167593        101521           60.58
    4      162669         93751           57.63
    5      179059        103200           57.63
    6      270670        171523           63.37
    7      332343        201223           60.55
    8      335352        185592           55.34
    9      261407        126795           48.50
   10      197940         83081           41.97
   11      175515         69155           39.40
   12      179402         71033           39.59
   13      207385         87846           42.36
   14      254392        118580           46.61
   15      284341        139165           48.94
   16      300950        150227           49.92
   17      340357        178986           52.59
   18      349075        190039           54.44
   19      329685        179698         

## 9. Figures

In [26]:
# Fig 1 - indexed monthly volume, CBD vs non-CBD, with the policy marker
m = monthly.copy()
m["t"] = (m["year"] - 2024) * 12 + m["month"]
fig, ax = plt.subplots(figsize=(8, 3))
for flag, lab in [(True, "CBD-touching"), (False, "non-CBD")]:
    s = m[m.cbd_trip == flag].sort_values("t")
    ax.plot(s["t"], s["trips"] / s["trips"].iloc[0] * 100, marker="o", ms=3, label=lab)
ax.axvline(13, color="crimson", ls="--", lw=1)
ax.annotate("congestion pricing\n5 Jan 2025", xy=(13, ax.get_ylim()[1]),
            xytext=(13.4, ax.get_ylim()[1] * 0.97), fontsize=8, color="crimson")
ax.set_xticks(range(1, 25, 3))
ax.set_xticklabels([f"{2024 + (i-1)//12}-{((i-1)%12)+1:02d}" for i in range(1, 25, 3)],
                   rotation=45)
ax.set_ylabel("trips, Jan 2024 = 100"); ax.legend()
ax.set_title("Monthly HVFHV volume, CBD-touching vs non-CBD")
fig.savefig("figures/p2/fig1_cbd_indexed_volume.png"); plt.close(fig)
print("saved fig1")

saved fig1


In [27]:
# Fig 2 - fee incidence by borough
fig, ax = plt.subplots(figsize=(6, 3))
d = inc.sort_values("fee_pct_of_fare")
ax.barh(d["pu_boro"], d["fee_pct_of_fare"], color="#2b6cb0")
for i, (v, f_) in enumerate(zip(d["fee_pct_of_fare"], d["mean_fare"])):
    ax.text(v + 0.02, i, f"mean fare ${f_:.2f}", va="center", fontsize=7)
ax.set_xlabel("congestion fee as % of base fare")
ax.set_title("Fee incidence by pickup borough, 2025")
fig.savefig("figures/p2/fig2_fee_incidence.png"); plt.close(fig)
print("saved fig2")

saved fig2


In [28]:
# Fig 3 - mismatch heatmap, worst 20 zones x hour
top = (zh_p.groupby("Zone")["mismatch_index"].mean().nlargest(20).index)
h = (zh_p[zh_p.Zone.isin(top)]
     .pivot_table(index="Zone", columns="hour", values="mismatch_index"))
h = h.loc[h.mean(axis=1).sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(h.values, aspect="auto", cmap="RdYlGn_r", vmin=0.6, vmax=1.6)
ax.set_yticks(range(len(h))); ax.set_yticklabels(h.index, fontsize=7)
ax.set_xticks(range(0, 24, 2)); ax.set_xlabel("hour of day")
ax.set_title("Service mismatch index (zone wait / citywide wait at same hour)")
ax.grid(False); fig.colorbar(im, ax=ax, label="mismatch index")
fig.savefig("figures/p2/fig3_mismatch_heatmap.png"); plt.close(fig)
print("saved fig3")

saved fig3


In [29]:
# Fig 4 - demand density vs wait
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.scatter(sub["log_trips"], sub["mean_wait_s"] / 60, s=3, alpha=0.25, c="#2b6cb0")
z = pd.Series(sub["mean_wait_s"] / 60).groupby(pd.cut(sub["log_trips"], 20)).mean()
ax.plot([i.mid for i in z.index], z.values, color="crimson", lw=1.6, label="binned mean")
ax.set_xlabel("log10 trips in zone-hour"); ax.set_ylabel("mean wait (min)")
ax.set_title(f"Demand density vs wait (r = {corr:.3f})"); ax.legend()
fig.savefig("figures/p2/fig4_density_vs_wait.png"); plt.close(fig)
print("saved fig4")

saved fig4


In [30]:
# Fig 5 - RDD vs DataFrame runtime
fig, ax = plt.subplots(figsize=(4.2, 2.6))
ax.bar(["RDD\nreduceByKey", "DataFrame\ngroupBy"], [rdd_secs, df_secs],
       color=["#718096", "#2b6cb0"])
for i, v in enumerate([rdd_secs, df_secs]):
    ax.text(i, v, f"{v:.1f}s", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("seconds"); ax.set_title("Identical aggregation, two APIs")
fig.savefig("figures/p2/fig5_rdd_vs_dataframe.png"); plt.close(fig)
print("saved fig5")

saved fig5


## 10. Numbers for the report

In [31]:
summary = {
    "records_2024": int(by_year.loc[by_year.year == 2024, "count"].iloc[0]),
    "records_2025": int(by_year.loc[by_year.year == 2025, "count"].iloc[0]),
    "records_total": n_total,
    "agg_rows": int(A.count()),
    "n_cbd_zones": len(cbd_ids),
    "cbd_threshold": CBD_THRESHOLD,
    "cbd_share_2024_pct": round(float(base24["cbd_share_pct"].mean()), 2),
    "cbd_share_2025_pct": round(float(base25["cbd_share_pct"].mean()), 2),
    "cbd_fee_total_2025": round(float(inc["fee"].sum()), 2),
    "rdd_seconds": round(rdd_secs, 1),
    "df_seconds": round(df_secs, 1),
    "speedup": round(speedup, 2),
    "density_wait_corr": round(float(corr), 3),
    "mismatch_p90_threshold": round(float(thr), 3),
}
json.dump(summary, open("output/p2/report_numbers_p2.json", "w"), indent=2)
for k, v in summary.items():
    print(f"{k:28s}: {v}")
print("\noutput/p2:", sorted(os.listdir("output/p2")))
print("figures/p2:", sorted(os.listdir("figures/p2")))

records_2024                : 229391984
records_2025                : 232463392
records_total               : 461855376
agg_rows                    : 73973527
n_cbd_zones                 : 38
cbd_threshold               : 0.8
cbd_share_2024_pct          : 34.84
cbd_share_2025_pct          : 32.88
cbd_fee_total_2025          : 117124705.5
rdd_seconds                 : 12.7
df_seconds                  : 1.9
speedup                     : 6.56
density_wait_corr           : -0.472
mismatch_p90_threshold      : 1.297

output/p2: ['agg_zone_hour_month.parquet', 'cbd_share_by_month.csv', 'cbd_zones.json', 'density_wait_corr_by_hour.csv', 'dual_implementation_timing.json', 'fee_incidence_by_borough.csv', 'monthly_cbd_vs_noncbd.csv', 'physical_plan_groupby.txt', 'report_numbers_p2.json', 'segment_monthly.csv', 'service_deserts.csv', 'shared_match_by_hour.csv', 'zone_charge_rate.csv', 'zone_hour_wait.csv']
figures/p2: ['fig1_cbd_indexed_volume.png', 'fig2_fee_incidence.png', 'fig3_mismatch_heatma

### Before you close the session

1. **Screenshot the Spark UI** — DAG visualisation, stage list with shuffle
   read/write, task timeline. It dies with the session.
2. Event logs persist in `D:\805\spark-events`; reopen later with
   `spark-class org.apache.spark.deploy.history.HistoryServer`.
3. Committing `output/p2/` and `figures/p2/` — not the aggregate Parquet if it is
   large; add `output/p2/agg_zone_hour_month.parquet/` to `.gitignore` if so.
4.  Savannah can access the shared aggregate ready at
   `output/p2/agg_zone_hour_month.parquet`.